# RAG 第 15 课：Query Rewriting 与 Multi-Query Expansion

前 14 课我们把"检索侧"基本拉平了：
- chunking / embedding / hybrid / rerank
- evaluation / LLM-as-a-Judge

但还有一个盲点：**我们一直默认用户给进来的 query 本身就是好的 query。**

真实项目里根本不是这样：
- 用户 query 太短
- 用户 query 含口语、拼错、指代
- 用户 query 含多个子问题
- 用户 query 的关键词和文档里写法不一样

所以这节课我们在检索之前加一步：**让 LLM 先把 query 变得更适合检索。**

具体做两种：
1. **Query Rewriting**：把 query 重写成一句更规范的问题
2. **Multi-Query Expansion**：让 LLM 生成多个角度的 query，一起去检索再融合

然后和 baseline（直接用原 query）对比，看到底哪种情况下改写有用。

In [ ]:
# 先清理 GPU 缓存。
import gc

try:
    import torch
    if torch.cuda.is_available():
        gc.collect()
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()
        print('GPU cache cleared.')
    else:
        print('CUDA not available, skip GPU cache clear.')
except Exception as e:
    print(f'Torch not available, skip GPU cache clear. ({e})')

## 1. 连接 LM Studio

本地 LLM 刚刚升级到 **Qwen 3.6**，端口没变。
这一格会自动发现，并优先选用 `qwen/qwen3.6-35b-a3b`，老模型作为 fallback。

In [ ]:
import json
import math
import re
from collections import Counter, defaultdict
from typing import List

import numpy as np
from datasets import load_dataset
from openai import OpenAI

BASE_URL = 'http://10.0.0.63:1234/v1'
API_KEY = 'lm-studio'

client = OpenAI(base_url=BASE_URL, api_key=API_KEY)
model_ids = [m.id for m in client.models.list().data]
print('available models:', model_ids)

# 优先用新发布的 Qwen 3.6
preferred_chat_models = ['qwen/qwen3.6-35b-a3b', 'qwen3.6-35b-a3b', 'qwen3.5-35b-a3b', 'qwen3.5-27b']
chat_model = next((m for m in preferred_chat_models if m in model_ids), None)
embedding_model = next((m for m in model_ids if 'embed' in m.lower() or 'embedding' in m.lower()), None)

print('chat_model      =', chat_model)
print('embedding_model =', embedding_model)

if chat_model is None:
    raise RuntimeError('没有找到可用的 chat model。')
if embedding_model is None:
    raise RuntimeError('没有找到可用的 embedding model。')

## 2. 加载真实数据并复用第 14 课的 hybrid 检索

因为 query rewriting 的效果要放在"能用的检索系统"上看，我们直接把第 14 课的 BM25 + dense + RRF 搬过来用。
你可以把这一段当作"检索层 SDK"。

In [ ]:
raw_ds = load_dataset('squad', split='validation[:160]')

context_to_doc_id = {}
documents = []
eval_rows = []

for item in raw_ds:
    context = item['context'].strip()
    if context not in context_to_doc_id:
        doc_id = len(documents)
        context_to_doc_id[context] = doc_id
        documents.append({'doc_id': doc_id, 'text': context, 'title': item['title']})
    else:
        doc_id = context_to_doc_id[context]

    if item['answers']['text']:
        eval_rows.append({
            'question': item['question'].strip(),
            'gold_doc_id': doc_id,
            'reference_answer': item['answers']['text'][0].strip(),
            'title': item['title'],
        })

print('num_documents =', len(documents))
print('num_eval_rows =', len(eval_rows))

In [ ]:
def tokenize(text: str) -> List[str]:
    return re.findall(r'[a-zA-Z0-9]+', text.lower())


class BM25:
    def __init__(self, corpus_tokens: List[List[str]], k1: float = 1.5, b: float = 0.75):
        self.k1 = k1
        self.b = b
        self.doc_lens = np.array([len(toks) for toks in corpus_tokens], dtype=np.float32)
        self.avgdl = float(self.doc_lens.mean()) if len(corpus_tokens) > 0 else 0.0
        self.N = len(corpus_tokens)

        self.postings = defaultdict(dict)
        for doc_id, toks in enumerate(corpus_tokens):
            tf = Counter(toks)
            for token, freq in tf.items():
                self.postings[token][doc_id] = freq

        self.idf = {}
        for token, posting in self.postings.items():
            df = len(posting)
            self.idf[token] = math.log(1 + (self.N - df + 0.5) / (df + 0.5))

    def score(self, query_tokens: List[str]) -> np.ndarray:
        scores = np.zeros(self.N, dtype=np.float32)
        for token in query_tokens:
            if token not in self.postings:
                continue
            idf = self.idf[token]
            for doc_id, freq in self.postings[token].items():
                dl = self.doc_lens[doc_id]
                denom = freq + self.k1 * (1 - self.b + self.b * dl / self.avgdl)
                scores[doc_id] += idf * (freq * (self.k1 + 1)) / denom
        return scores


doc_tokens = [tokenize(doc['text']) for doc in documents]
bm25 = BM25(doc_tokens)
print('bm25 vocab size =', len(bm25.postings))

In [ ]:
def get_embeddings(texts: List[str], batch_size: int = 16) -> np.ndarray:
    all_vectors = []
    for start in range(0, len(texts), batch_size):
        batch = texts[start:start + batch_size]
        response = client.embeddings.create(model=embedding_model, input=batch)
        batch_vectors = [np.array(item.embedding, dtype=np.float32) for item in response.data]
        all_vectors.extend(batch_vectors)
    return np.vstack(all_vectors)


def l2_normalize(matrix: np.ndarray) -> np.ndarray:
    norms = np.linalg.norm(matrix, axis=1, keepdims=True)
    norms = np.clip(norms, 1e-12, None)
    return matrix / norms


doc_texts = [doc['text'] for doc in documents]
doc_embeddings = l2_normalize(get_embeddings(doc_texts, batch_size=16))
print('doc_embeddings shape =', doc_embeddings.shape)

In [ ]:
def lexical_retrieve(question: str, top_k: int = 10):
    scores = bm25.score(tokenize(question))
    top_indices = np.argsort(scores)[::-1][:top_k]
    return [(int(idx), float(scores[idx])) for idx in top_indices]


def dense_retrieve(question: str, top_k: int = 10):
    q_emb = l2_normalize(get_embeddings([question], batch_size=1))[0]
    scores = doc_embeddings @ q_emb
    top_indices = np.argsort(scores)[::-1][:top_k]
    return [(int(idx), float(scores[idx])) for idx in top_indices]


def rrf_fuse(rankings_list, k: int = 60, top_k: int = 10):
    fused = defaultdict(float)
    for rankings in rankings_list:
        for rank, (doc_id, _) in enumerate(rankings, start=1):
            fused[doc_id] += 1.0 / (k + rank)
    sorted_items = sorted(fused.items(), key=lambda x: x[1], reverse=True)
    return [(doc_id, score) for doc_id, score in sorted_items[:top_k]]


def hybrid_retrieve(question: str, top_k: int = 10, candidate_k: int = 20):
    lex = lexical_retrieve(question, top_k=candidate_k)
    dense = dense_retrieve(question, top_k=candidate_k)
    return rrf_fuse([lex, dense], k=60, top_k=top_k)

## 3. Query Rewriting：一次性重写

最简单的一种 query 变换：让 LLM 把用户的问题重写成一句更规范、更适合检索的问题。

常见的目标是：
- 补全主语 / 指代
- 把口语改成标准写法
- 展开缩写、纠正拼写
- 保留关键实体，别画蛇添足

注意 prompt 里要**明确禁止 LLM 自由发挥**，否则它会替你改变问题的含义，检索反而更差。

In [ ]:
def rewrite_query(question: str) -> str:
    system_prompt = (
        'You rewrite user questions into a single cleaner search query. '
        'Rules: (1) preserve the original intent exactly; '
        '(2) keep all named entities, numbers and proper nouns; '
        '(3) expand pronouns and common abbreviations only when obvious; '
        '(4) do not add facts that are not in the original question; '
        '(5) output the rewritten query only, no explanations.'
    )
    user_prompt = f'Original question: {question}\n\nRewritten query:'
    response = client.chat.completions.create(
        model=chat_model,
        messages=[
            {'role': 'system', 'content': system_prompt},
            {'role': 'user', 'content': user_prompt},
        ],
        temperature=0,
    )
    return response.choices[0].message.content.strip().strip('"').strip("'")


demo_q = eval_rows[0]['question']
print('original :', demo_q)
print('rewritten:', rewrite_query(demo_q))

## 4. Multi-Query Expansion：一次生成多个 query

另一种思路更有意思：**让 LLM 从不同角度生成 N 个 query**。

它的好处是：
- 不同 query 更容易覆盖不同写法的文档
- 每个 query 各自去检索，再融合结果
- 哪怕某一个 query 写歪了，其他 query 还能救回来

融合我们依然用 RRF。这次我们把"原 query + N 个改写 query"的检索结果一起喂给 RRF。

In [ ]:
def generate_query_variants(question: str, n: int = 3) -> List[str]:
    system_prompt = (
        'You generate alternative search queries for a retrieval system. '
        'Given an original question, produce diverse paraphrases that keep the same meaning '
        'but vary in wording, entity forms, and the balance between keywords and natural language. '
        'Do not invent new facts. Do not answer the question. '
        'Return strict JSON with a single key "queries" whose value is a list of strings.'
    )
    user_prompt = (
        f'Original question: {question}\n'
        f'Produce exactly {n} alternative queries.'
    )
    response = client.chat.completions.create(
        model=chat_model,
        messages=[
            {'role': 'system', 'content': system_prompt},
            {'role': 'user', 'content': user_prompt},
        ],
        temperature=0.4,  # 需要一点多样性
    )
    raw = response.choices[0].message.content.strip()
    match = re.search(r'\{.*\}', raw, flags=re.S)
    if not match:
        # 兜底：把 raw 按行切
        lines = [l.strip('-*. \t').strip() for l in raw.splitlines() if l.strip()]
        return [l for l in lines if l][:n]
    parsed = json.loads(match.group(0))
    queries = parsed.get('queries', [])
    return [str(q).strip() for q in queries if str(q).strip()][:n]


variants = generate_query_variants(demo_q, n=3)
print('original:', demo_q)
for i, v in enumerate(variants, start=1):
    print(f'variant {i}: {v}')

## 5. 三种检索策略统一封装

为了后面比较方便，我们把三种策略都封装成"输入 question，输出 top_k 个 doc_id"：

In [ ]:
def retrieve_baseline(question: str, top_k: int = 3):
    return [doc_id for doc_id, _ in hybrid_retrieve(question, top_k=top_k)]


def retrieve_with_rewrite(question: str, top_k: int = 3):
    rewritten = rewrite_query(question)
    return [doc_id for doc_id, _ in hybrid_retrieve(rewritten, top_k=top_k)], rewritten


def retrieve_with_multi_query(question: str, top_k: int = 3, n_variants: int = 3, per_query_k: int = 10):
    variants = generate_query_variants(question, n=n_variants)
    all_queries = [question] + variants
    rankings_list = [hybrid_retrieve(q, top_k=per_query_k) for q in all_queries]
    fused = rrf_fuse(rankings_list, k=60, top_k=top_k)
    return [doc_id for doc_id, _ in fused], variants

## 6. 接上 LLM 生成答案 + 评估函数

这部分和前几课一样，照搬过来：

In [ ]:
def build_context_block(doc_ids):
    parts = []
    for i, doc_id in enumerate(doc_ids, start=1):
        doc = documents[doc_id]
        parts.append(f'[Document {i}] title: {doc["title"]}\n{doc["text"]}')
    return '\n\n'.join(parts)


def answer_with_llm(question: str, doc_ids):
    system_prompt = (
        'You are a careful question-answering assistant. '
        'Answer only from the provided context. '
        'If the answer is not supported by the context, reply with NOT_FOUND. '
        'Keep the answer short.'
    )
    user_prompt = (
        f'Question: {question}\n\n'
        f'Context:\n{build_context_block(doc_ids)}\n\n'
        'Return only the answer.'
    )
    response = client.chat.completions.create(
        model=chat_model,
        messages=[
            {'role': 'system', 'content': system_prompt},
            {'role': 'user', 'content': user_prompt},
        ],
        temperature=0,
    )
    return response.choices[0].message.content.strip()


def normalize_token(token):
    token = token.lower()
    token = re.sub(r'[^a-z0-9]+', '', token)
    return token


def to_tokens(text):
    return [t for t in (normalize_token(x) for x in text.split()) if t]


def normalize_text(text):
    return ' '.join(to_tokens(text))


def exact_match(prediction, reference):
    return 1.0 if normalize_text(prediction) == normalize_text(reference) else 0.0


def token_f1(prediction, reference):
    pred_tokens = to_tokens(prediction)
    ref_tokens = to_tokens(reference)
    pred_counter = Counter(pred_tokens)
    ref_counter = Counter(ref_tokens)
    overlap = sum((pred_counter & ref_counter).values())
    if overlap == 0:
        return 0.0
    precision = overlap / len(pred_tokens)
    recall = overlap / len(ref_tokens)
    return 2 * precision * recall / (precision + recall)

## 7. 先看单条样本

这一格会一眼看出三种策略的差别：

In [ ]:
sample = eval_rows[0]

base_ids = retrieve_baseline(sample['question'], top_k=3)
rw_ids, rewritten = retrieve_with_rewrite(sample['question'], top_k=3)
mq_ids, variants = retrieve_with_multi_query(sample['question'], top_k=3, n_variants=3)

print('question:', sample['question'])
print('gold_doc_id:', sample['gold_doc_id'])
print('reference_answer:', sample['reference_answer'])
print()
print('baseline top-3      :', base_ids)
print('rewrite top-3       :', rw_ids, '   rewritten=', rewritten)
print('multi-query top-3   :', mq_ids)
for i, v in enumerate(variants, start=1):
    print(f'   variant {i}: {v}')

print()
print('baseline answer    :', answer_with_llm(sample['question'], base_ids))
print('rewrite  answer    :', answer_with_llm(sample['question'], rw_ids))
print('multi-query answer :', answer_with_llm(sample['question'], mq_ids))

## 8. 小规模批量评估

每条样本会调用 LLM 多次（rewrite 1 次、multi-query 生成 1 次、生成答案 3 次），所以先跑前 6 条。
你确认跑得通后可以自己加大。

In [ ]:
eval_subset = eval_rows[:6]
results = []

for i, row in enumerate(eval_subset, start=1):
    print(f'processing {i}/{len(eval_subset)}: {row["question"]}')

    base_ids = retrieve_baseline(row['question'], top_k=3)
    try:
        rw_ids, rewritten = retrieve_with_rewrite(row['question'], top_k=3)
    except Exception as e:
        print('  rewrite failed, fallback to baseline:', e)
        rw_ids, rewritten = base_ids, row['question']

    try:
        mq_ids, variants = retrieve_with_multi_query(row['question'], top_k=3, n_variants=3)
    except Exception as e:
        print('  multi-query failed, fallback to baseline:', e)
        mq_ids, variants = base_ids, []

    ans_base = answer_with_llm(row['question'], base_ids)
    ans_rw = answer_with_llm(row['question'], rw_ids)
    ans_mq = answer_with_llm(row['question'], mq_ids)

    results.append({
        'question': row['question'],
        'gold_doc_id': row['gold_doc_id'],
        'reference_answer': row['reference_answer'],
        'rewritten': rewritten,
        'variants': variants,
        'base_ids': base_ids,
        'rw_ids': rw_ids,
        'mq_ids': mq_ids,
        'base_hit@1': 1.0 if base_ids and base_ids[0] == row['gold_doc_id'] else 0.0,
        'rw_hit@1': 1.0 if rw_ids and rw_ids[0] == row['gold_doc_id'] else 0.0,
        'mq_hit@1': 1.0 if mq_ids and mq_ids[0] == row['gold_doc_id'] else 0.0,
        'base_hit@3': 1.0 if row['gold_doc_id'] in base_ids else 0.0,
        'rw_hit@3': 1.0 if row['gold_doc_id'] in rw_ids else 0.0,
        'mq_hit@3': 1.0 if row['gold_doc_id'] in mq_ids else 0.0,
        'ans_base': ans_base,
        'ans_rw': ans_rw,
        'ans_mq': ans_mq,
        'em_base': exact_match(ans_base, row['reference_answer']),
        'em_rw': exact_match(ans_rw, row['reference_answer']),
        'em_mq': exact_match(ans_mq, row['reference_answer']),
        'f1_base': token_f1(ans_base, row['reference_answer']),
        'f1_rw': token_f1(ans_rw, row['reference_answer']),
        'f1_mq': token_f1(ans_mq, row['reference_answer']),
    })

summary = {
    'Hit@1 ': {
        'baseline   ': float(np.mean([x['base_hit@1'] for x in results])),
        'rewrite    ': float(np.mean([x['rw_hit@1'] for x in results])),
        'multi-query': float(np.mean([x['mq_hit@1'] for x in results])),
    },
    'Hit@3 ': {
        'baseline   ': float(np.mean([x['base_hit@3'] for x in results])),
        'rewrite    ': float(np.mean([x['rw_hit@3'] for x in results])),
        'multi-query': float(np.mean([x['mq_hit@3'] for x in results])),
    },
    'EM    ': {
        'baseline   ': float(np.mean([x['em_base'] for x in results])),
        'rewrite    ': float(np.mean([x['em_rw'] for x in results])),
        'multi-query': float(np.mean([x['em_mq'] for x in results])),
    },
    'F1    ': {
        'baseline   ': float(np.mean([x['f1_base'] for x in results])),
        'rewrite    ': float(np.mean([x['f1_rw'] for x in results])),
        'multi-query': float(np.mean([x['f1_mq'] for x in results])),
    },
}

print('\n=== summary ===')
for metric, sub in summary.items():
    print(metric, sub)

## 9. 看逐条结果

老规矩，逐条比汇总更有信息量。
特别关注这几类样本：
- baseline 没命中但 rewrite / multi-query 命中的
- baseline 命中但改写之后反而掉的（**query rewriting 把问题带歪了**）
- 三种都命中的（这时改写并没有额外收益，只是多花了 API 调用）

In [ ]:
for row in results:
    print('question    :', row['question'])
    print('rewritten   :', row['rewritten'])
    print('variants    :', row['variants'])
    print('gold_doc_id :', row['gold_doc_id'])
    print('base_ids    :', row['base_ids'])
    print('rw_ids      :', row['rw_ids'])
    print('mq_ids      :', row['mq_ids'])
    print(f"hit@1 base/rw/mq: {row['base_hit@1']}/{row['rw_hit@1']}/{row['mq_hit@1']}")
    print(f"hit@3 base/rw/mq: {row['base_hit@3']}/{row['rw_hit@3']}/{row['mq_hit@3']}")
    print('reference   :', row['reference_answer'])
    print('ans_base    :', row['ans_base'])
    print('ans_rw      :', row['ans_rw'])
    print('ans_mq      :', row['ans_mq'])
    print(f"EM  base/rw/mq: {row['em_base']}/{row['em_rw']}/{row['em_mq']}")
    print(f"F1  base/rw/mq: {round(row['f1_base'],3)}/{round(row['f1_rw'],3)}/{round(row['f1_mq'],3)}")
    print('-' * 100)

## 10. 这节课的工程直觉

Query rewriting 不是银弹，真实项目里它的收益差异非常大，取决于：

1. **原始 query 的质量**：对 squad 这种"已经写得很清楚"的 query，改写收益往往有限。对用户在真实产品里打出来的含糊 query，收益会更明显。
2. **改写约束是否足够严格**：如果 prompt 太开放，LLM 会偷偷改变语义，反而降低检索质量。
3. **Multi-Query 的代价**：它会让调用次数至少翻一倍，所以适合"一次就要答对"的场景，不适合高 QPS 的场景。
4. **Multi-Query 比 Rewrite 更稳**：因为它保留了原 query，最差情况不会比 baseline 更差。

所以工程上常见的一种折中是：

**只在第一次检索不够自信（命中分数低 / top-k 分数太接近）时，才触发 rewrite 或 multi-query。**

## 11. 本课小结

这节课你要带走的核心是：

1. 检索之前先动一下 query，往往比换一个更大的 embedding 模型更划算。
2. Query Rewriting 是最轻量的一种做法，关键是 prompt 里要严格约束 LLM。
3. Multi-Query Expansion 是更稳的一种做法，代价是多跑几次检索。
4. 融合依然推荐 RRF：天然适合把多路检索结果合并。
5. 真实系统里 query rewriting 往往是**条件触发**的，不是每次都用。

下一课最自然的衔接就是：
**HyDE（Hypothetical Document Embeddings）**，也就是让 LLM 先写一个"假设的答案"当作 query 再去检索。
你会看到它和 multi-query 的味道很接近，但思路完全不同。